# IOI + the Hydra effect in GPT-2 small

**Question.** GPT-2 small solves Indirect Object Identification (IOI) with a known circuit ([Wang et al. 2022](https://arxiv.org/abs/2211.00593)).
When we ablate the heads that *directly* write the answer (the name movers), how much of the loss does the rest of the network make up for
([McGrath et al. 2023, the Hydra effect](https://arxiv.org/abs/2307.15771)), which heads do the repairing, and how much do the answers depend on
the choice of ablation (a systematic uncertainty)?

**Plan.**
1. Dataset and baseline, with bootstrap CIs over prompts
2. Residual-stream activation patching (layer × position)
3. Per-head activation patching: recover the circuit
4. Direct logit attribution (DLA) and attention patterns: name movers vs negative name movers
5. **Hydra:** naive direct-effect prediction vs actual drop, and who compensates
6. **Additivity:** single-head ablation drops vs joint ablation drop
7. **Systematics:** ablation method, template, dataset seed
8. Write-up

Helpers live in `ioi_hydra.py`. Everything is computed per prompt so it can be bootstrapped.
The full notebook runs in about 5 minutes on CPU on an M-series Mac; the head-patching cell is the slow one.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, torch, matplotlib.pyplot as plt
from transformer_lens.model_bridge import TransformerBridge
from ioi_hydra import *
from pathlib import Path

FIGDIR = Path("../docs/figures")  # figures used by docs/results.md
FIGDIR.mkdir(parents=True, exist_ok=True)
savefig = lambda name: plt.savefig(FIGDIR / f"{name}.png", dpi=150, bbox_inches="tight")

DEVICE = "cpu"  # "mps" also works on Apple silicon; CPU is plenty for GPT-2 small
model = TransformerBridge.boot_transformers("gpt2", device=DEVICE)
model.enable_compatibility_mode()  # fold LayerNorm, etc. -> numerics match legacy HookedTransformer / the papers
L, H = model.cfg.n_layers, model.cfg.n_heads

## 1. Dataset and baseline

8 templates × 16 prompts, balanced between ABBA and BABA name orders. Each clean prompt has a matched **ABC** corrupt prompt
(same template, three fresh names), which carries no information about which name is the IO.
Metric: logit difference LD = logit(IO) − logit(S) at the final token. The IOI paper reports about 3.5 on its own dataset.

In [ ]:
ds = make_ioi_dataset(model, n_per_template=16, seed=0)
for i in range(4):
    print(f"{ds.order[i]}  {ds.prompts[i]!r:80}  ->  {model.to_string(ds.io[i])!r}")
    print(f"      {ds.corrupt_prompts[i]!r}")

clean_logits, clean_cache = model.run_with_cache(ds.clean_tokens)
corrupt_logits, corrupt_cache = model.run_with_cache(ds.corrupt_tokens)
clean_ld, corrupt_ld = logit_diff(clean_logits, ds), logit_diff(corrupt_logits, ds)
CLEAN, CORRUPT = clean_ld.mean().item(), corrupt_ld.mean().item()

print(f"\nclean LD   {fmt_ci(bootstrap_ci(clean_ld))}   (fraction LD > 0: {(clean_ld > 0).float().mean():.3f})")
print(f"corrupt LD {fmt_ci(bootstrap_ci(corrupt_ld))}")

In [ ]:
# Template-to-template spread is a systematic: is it larger than the statistical error on each template?
for t, tmpl in enumerate(TEMPLATES):
    print(f"{t}  LD {fmt_ci(bootstrap_ci(clean_ld[ds.template_id == t]))}   {tmpl}")

## 2. Residual-stream patching (layer × position)

Denoising: run the **corrupt** prompt, patch in the **clean** `resid_pre` at one (layer, position), and measure normalized LD
(0 = corrupt, 1 = clean). Uses a single template so that positions line up.

*Expect:* early on the information sits at the names (IO, S1, S2). Around layers 7–9 it moves to the final token.

In [ ]:
ds0 = ds.subset(ds.template_id == 0)
_, cache0 = model.run_with_cache(ds0.clean_tokens)
resid_patch = patch_resid_by_pos(model, ds0, cache0, CLEAN, CORRUPT)
labels = [f"{i}:{t}" for i, t in enumerate(model.to_str_tokens(ds0.clean_tokens[0]))]
heatmap(resid_patch, "Residual patching (normalized LD)", xlabel="position", xticklabels=labels, vmax=1);
savefig("resid_patching")

## 3. Per-head patching

Patch each head's clean output (`hook_z`, all positions) into the corrupt run. This takes about 2 minutes on CPU.

*Compare with the paper:* name movers 9.9, 9.6, 10.0; S-inhibition 7.3, 7.9, 8.6, 8.10; induction / duplicate-token heads in
layers 0–6; backup name movers 10.10, 10.6, 10.2, 10.1, 11.2, 9.7, 9.0, 11.9; negative name movers 10.7, 11.10.

In [ ]:
head_patch = patch_heads(model, ds, clean_cache, CLEAN, CORRUPT)   # [N, L, H]
heatmap(head_patch.mean(0), "Head patching: clean hook_z -> corrupt run (normalized LD)", annotate_top=10)
for l, h, v in top_heads(head_patch.mean(0), 10):
    print(f"{l}.{h:<3} {fmt_ci(bootstrap_ci(head_patch[:, l, h]), 3)}")
savefig("head_patching")

## 4. Direct logit attribution

Each head's direct contribution to LD at the final token (output → final LN scale → unembed). Name movers should be strongly positive
and attend END → IO. Negative name movers should be negative and also attend to IO, pushing *down* the name they copy.

In [ ]:
de_clean = head_direct_effects(model, clean_cache, ds)    # [N, L, H]
heatmap(de_clean.mean(0), "Direct effect on LD per head (clean run)", annotate_top=10)

print("head   DE                     attn END->IO   attn END->S1")
for l, h, _ in top_heads(de_clean.mean(0).abs(), 8):
    print(f"{l}.{h:<4} {fmt_ci(bootstrap_ci(de_clean[:, l, h])):22} "
          f"{attention_to(clean_cache, ds, l, h, 'IO').mean():.2f}           "
          f"{attention_to(clean_cache, ds, l, h, 'S1').mean():.2f}")
savefig("direct_effects")

## 5. The Hydra effect

Ablate the three name movers. If the network were a sum of independent parts, LD would drop by the **direct effect** those heads lose
(the naive prediction). The **total effect** is what actually happens. Their gap is the **repair**:

$$\text{repair} = \underbrace{\textstyle\sum_{h \in \text{ablated}} (DE_h^{\text{clean}} - DE_h^{\text{abl}})}_{\text{direct}} - \underbrace{(LD^{\text{clean}} - LD^{\text{abl}})}_{\text{total}}$$

Note: DE uses each run's own final-LN scale, so "repair" includes LayerNorm rescaling as well as heads changing their outputs. Separating
the two (by freezing the LN scale) is a good extension.

In [ ]:
ablator = Ablator(model, ds, corrupt_cache)
NAME_MOVERS = [(9, 9), (9, 6), (10, 0)]

r = hydra(ablator, NAME_MOVERS, "mean", clean_ld, de_clean)
print("direct (naive prediction)", fmt_ci(bootstrap_ci(r["direct"])))
print("total  (actual drop)     ", fmt_ci(bootstrap_ci(r["total"])))
print("repair fraction          ", fmt_ci(bootstrap_ci(r["repair"], r["direct"], stat=lambda a, b: a.mean() / b.mean())))

heatmap(r["delta_de"].mean(0), "Change in direct effect after mean-ablating name movers", annotate_top=8)
print("\ntop compensators (ΔDE):")
for l, h, v in top_heads(r["delta_de"].mean(0), 8):
    print(f"  {l}.{h:<3} {fmt_ci(bootstrap_ci(r['delta_de'][:, l, h]))}")
savefig("hydra_delta_de")

**Things to interpret:**
- How much of the repair comes from the *negative* name mover 10.7 becoming less negative, and how much from backup name movers becoming more positive?
  (Compare McDougall et al. 2023, [Copy Suppression](https://arxiv.org/abs/2310.04625).)
- Summed over all heads, does ΔDE account for the repair, or is a big share in MLPs and LN scale?

## 6. Additivity

Two different ways the network can be non-additive:
- **direct ≠ total** (§5): the network adjusts *downstream* of the ablation.
- **Σ singles ≠ joint**: ablating heads one at a time does not add up to ablating them together, so the heads interact.

Interaction = Σ singles − joint, bootstrapped per prompt (paired).

In [ ]:
HEAD_SETS = {
    "name movers": NAME_MOVERS,
    "NM + negative NM": NAME_MOVERS + [(10, 7), (11, 10)],
    "NM + backup NM": NAME_MOVERS + [(10, 10), (10, 6), (10, 2), (10, 1), (11, 2), (9, 7), (9, 0), (11, 9)],
    "S-inhibition": [(7, 3), (7, 9), (8, 6), (8, 10)],
}
rows = []
for name, heads in HEAD_SETS.items():
    for kind in ABLATION_KINDS:
        a = additivity(ablator, heads, kind, clean_ld)
        s, j = a["singles"].sum(-1), a["joint"]
        rows.append((name, kind, bootstrap_ci(s), bootstrap_ci(j), bootstrap_ci(s - j)))
        print(f"{name:18} {kind:9} Σsingles {fmt_ci(rows[-1][2]):22} joint {fmt_ci(rows[-1][3]):22} interaction {fmt_ci(rows[-1][4])}")

## 7. Systematics

**(a) Ablation method.** Zero, mean (per template and position, over ABC prompts), and resample (the matched ABC prompt) ask different
counterfactual questions. Zero ablation pushes activations off-distribution. Treat the spread as a systematic uncertainty on the headline number.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
w = 0.38
for i, kind in enumerate(ABLATION_KINDS):
    rk = hydra(ablator, NAME_MOVERS, kind, clean_ld, de_clean)
    for off, key, color in [(-w / 2, "direct", "C0"), (w / 2, "total", "C3")]:
        p, lo, hi = bootstrap_ci(rk[key])
        ax.bar(i + off, p, w, yerr=[[p - lo], [hi - p]], color=color, capsize=3, label=key if i == 0 else None)
ax.axhline(0, color="k", lw=0.5)
ax.set(xticks=range(len(ABLATION_KINDS)), xticklabels=ABLATION_KINDS, ylabel="drop in LD",
       title="Ablating name movers: predicted (direct) vs actual (total)")
ax.legend();
savefig("direct_vs_total")

**(b) Template** and **(c) dataset seed**: does the repair fraction move by more than its statistical error?

In [ ]:
frac = lambda a, b: a.mean() / b.mean()
r = hydra(ablator, NAME_MOVERS, "mean", clean_ld, de_clean)
for t in range(len(TEMPLATES)):
    m = ds.template_id == t
    print(f"template {t}: repair fraction {fmt_ci(bootstrap_ci(r['repair'][m], r['direct'][m], stat=frac))}")

In [ ]:
for seed in range(1, 4):
    ds_s = make_ioi_dataset(model, n_per_template=16, seed=seed)
    cl_s, cc_s = model.run_with_cache(ds_s.clean_tokens)
    _, kc_s = model.run_with_cache(ds_s.corrupt_tokens)
    ld_s = logit_diff(cl_s, ds_s)
    r_s = hydra(Ablator(model, ds_s, kc_s), NAME_MOVERS, "mean", ld_s, head_direct_effects(model, cc_s, ds_s))
    print(f"seed {seed}: clean LD {ld_s.mean():.2f}  repair fraction {fmt_ci(bootstrap_ci(r_s['repair'], r_s['direct'], stat=frac))}")

## 8. Write-up (fill in)

- **Headline:** Mean-ablating GPT-2 small's three IOI name-mover heads removes ___ ± ___ of direct logit difference, but the actual
  logit difference drops by only ___ ± ___. The network repairs ___% of the damage.
- **Who repairs:** ___ (10.7, the negative name mover, contributes ___; backup name movers contribute ___).
- **Interactions:** Σ singles vs joint: ___
- **Systematics:** ablation method moves the repair fraction by ___, templates by ___, seeds by ___ (statistical error: ___).
- **So what:** a single-head ablation (or a direct effect alone) can badly misstate a component's importance. For Goodfire-style work
  (probes as rewards, Silico running autonomous ablation studies), effect sizes need both the counterfactual choice and the
  network's compensation reported alongside them.
- **Next:** freeze the LN scale to separate LN rescaling from head compensation; path patching to see *how* 10.7 detects the ablation; repeat on Pythia or Gemma-2-2B.